## **Unit FoV Visualisation  |  Catalogue VS Reconstructed Sources**

In [ ]:
from pathlib import Path
from typing import Any, NamedTuple, Optional

import numpy as np
from numpy.typing import NDArray
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from astropy.io.fits.fitsrec import FITS_rec

from bloodmoon.coords import shift2angle
from bloodmoon.io import simulation_files
from bloodmoon.mask import CodedMaskCamera, codedmask
from bloodmoon.mask import count, decode, variance
from bloodmoon.optim import model_shadowgram, model_sky
from bloodmoon.types import CoordEquatorial
import darksun as ds
from darksun.data import Log, DataLoader, CatalogueLoader

from IROSrec.handle import config_dirpaths
import imgmaker as mgm

plt.rcdefaults()

RCPARAMS = {
    'axes.formatter.limits': (-3, 3),
    'savefig.bbox': 'tight',
}
mpl.rcParams.update(RCPARAMS)
ds.show.set_figures_darkbkg()

In [ ]:
# MASK_FITS: str = "mask_NTHT_20260129_CORRECTED.fits"
MASK_FITS: str = "mask_NTHT_20250725.fits"

# SKYFIELD: str = "IROSDummy"
# DATA_FITS: str = "baseline_2-50keV_1ks"
# DATA_FITS: str = "iros_benchmark_2-50keV_mask_050_1040x17_1ks_mask25"
# RUN_ID: str = 'IROSbncmrk_detected_1ks_upx2_NewSGNorm_mask25'

SKYFIELD: str = "GalacticCentre"
DATA_FITS: str = "galctr_rxte-sax_mask_050_1040x17_2-50keV_1ks_mask25"

RUN_ID: str = 'GC_smoothed_rec_1ks_2-50keV_mask25'
# RUN_ID: str = 'GC_smoothed_rec_1ks_detected_2-6keV_mask25'
# RUN_ID: str = 'GC_rec_1ks_2-6keV_mask25'

ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"
DATASET: str = "reconstructed"

E_min: float = 2.0  # [keV]
E_max: float = 50.0  # [keV]
coords2exclude: list[CoordEquatorial] | None = None

UPS_X, UPS_Y = 2, 1
hide_bulk_els_y: float = 1.5   # [mm]

In [ ]:
MASK_PATH, SIMUL_DATA_PATH, SAVE_PATH = config_dirpaths(
    mask=MASK_FITS,
    skyfield=SKYFIELD,
    simul=DATA_FITS,
    runID=RUN_ID,
)
OUT_RESULTS_PATH = mgm.config_savedata_to()

wfm: CodedMaskCamera = codedmask(MASK_PATH, UPS_X, UPS_Y, hide_bulk_els_y=hide_bulk_els_y)

filepaths: dict[str, dict[str, Path]] = simulation_files(SIMUL_DATA_PATH)
sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET], E_min=E_min, E_max=E_max, coords=coords2exclude)
catA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])
sdlB = ds.get_data(filepaths[ID_CAMERA_B][DATASET], E_min=E_min, E_max=E_max, coords=coords2exclude)
catB = ds.get_catalogue(filepaths[ID_CAMERA_B]['sources'])

logA, logB = ds.load_database(f"{SAVE_PATH}/IROS_sources_db.fits")

In [ ]:
class Source(NamedTuple):
    """Container for info related to a source (ID, RA/Dec [deg], flux [ph/cm2/s])."""
    ID: str
    ra: float
    dec: float
    flux: float

def retrieve_detectable_srcs(catalogue: CatalogueLoader, exposure: float) -> FITS_rec:
    """Retrieves all detectable sources during a given exposure within the FoV of a LEM-X camera."""
    # extract min detectable flx @ 5sigma in given exposure by considering an off-axis source at
    # the edge of the instrumental FoV and the sensitivity map in Evangelista+2026 (Fig 8, pag 16)
    # @ https://link.springer.com/content/pdf/10.1007/s10686-026-10047-x.pdf
    #   * 2.5 Crab is the flux we need to observe a source @ 5sigma, edge FoV
    #   * 1 s / exposure is the conversion factor accounting for the exposure (sqrt due to SNR computation)
    #   * 2.7 Crab / ph/cm2/s is the conversion factor between instrumental flux and emitted flux in 2-50 keV
    min_flx = 2.5 / 2.7 * np.sqrt(1 / exposure)
    # select detectable sources
    mask = catalogue.DLdata['FLX'] > min_flx
    return catalogue.DLdata[mask]

def get_detectable_srcs_log(rec: FITS_rec) -> Log:
    """Builds the Log container for all detectable sources within camera FoV."""
    return

def get_detectable_srcs_data(log, camera, sdl) -> ...:
    """Extracts detectable sources data."""
    return

def show_skyfield_map(
    log: Log,
    camera: CodedMaskCamera,
    show_IDs: bool = True,
    rec_log: Optional[Log] = None,
    save_to: Optional[str | Path] = None,
    save_options: Optional[dict[str, Any]] = None,
    **fig_kws,
) -> None:
    """
    Displays sources in the camera FoV.
    """
    _skyx, _skyy = camera.bins_sky
    skyx, skyy = map(
        lambda x: shift2angle(camera, x),
        (_skyx, _skyy),
    )
    SETUP = {
        'x': 'angle_x',
        'y': 'angle_y',
        'xlabel': '$\\theta_{fine}$ [deg]',
        'ylabel': '$\\theta_{coarse}$ [deg]',
        'title': f'{log.name} Local Frame SkyMap Grid',

        'ancor': (skyx[0], skyy[0]),
        'width': skyx[-1] - skyx[0],
        'height': skyy[-1] - skyy[0],
        'color': '#120E16',
        'txt_color': 'white',
        'rot': 35,
        'alpha': 0.75,

        'xline': ((skyx[0], skyx[-1]), (0, 0)),
        'yline': ((0, 0), (skyy[0], skyy[-1])),

        'xlim': (skyx[0], skyx[-1]),
        'ylim': (skyy[0], skyy[-1]),
    }

    fig, ax = plt.subplots(1, 1, tight_layout=True, **fig_kws)
    ax.set_xlabel(xlabel=SETUP['xlabel'])
    ax.set_ylabel(ylabel=SETUP['ylabel'])
    ax.set_title(SETUP['title'])

    # plot bkg and axis lines
    ax.add_patch(
        Rectangle(
            xy=SETUP['ancor'], width=SETUP['width'], height=SETUP['height'],
            linewidth=0.1, edgecolor=SETUP['color'], facecolor=SETUP['color'],
        )
    )
    for line in ('xline', 'yline'):
        ax.plot(*SETUP[line], color="#4E4E4E", linestyle='-', linewidth=0.6, alpha=0.6)
    
    # plot sources on grid
    ax.scatter(
        log.log[SETUP['x']], log.log[SETUP['y']], c=None, label='detectable',
        edgecolor='DeepSkyBlue', alpha=SETUP['alpha'], s=10,
    )
    if rec_log is not None:
        ax.scatter(
            rec_log.log[SETUP['x']], rec_log.log[SETUP['y']], c=None, label='reconstr.',
            edgecolor='Orange', alpha=SETUP['alpha'], s=10,
        )
    if show_IDs:
        for name, x, y in zip(
            log.log['ID'],
            log.log[SETUP['x']],
            log.log[SETUP['y']],
        ):
            ax.text(x + 0.475, y + 0.385, name, color=SETUP['txt_color'], rotation=SETUP['rot'])

    ax.set_xlim(SETUP['xlim']); ax.set_ylim(SETUP['ylim'])
    ax.xaxis.set_ticks_position('both'); ax.yaxis.set_ticks_position('both')
    ax.tick_params(which='both', direction='in', width=1.5, length=4 if 'major' else 2.5)
    if rec_log is not None:
        ax.legend(loc='best')

    if save_to is not None:
        save_options_ = save_options or {}
        fig.savefig(save_to, **save_options_)
    
    plt.show()
    return